因子构建

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from lightgbm import LGBMRegressor

In [3]:
mnth_data=pd.read_csv('mnth_data.csv')

In [4]:
mnth_data['ts_code'] = mnth_data['ts_code'].astype(str).str.zfill(6)
mnth_data

,Unnamed: 0,ts_code,trade_month,close,circ_mv,total_mv,return_pct,excess_return,year_month,year,annual_rp,annual_rf,risk_free_rate,monthly_excess_return
0,0,000002,1999-01,7.85,2235682.38,3024264.91,-0.069905,0.250319,1999-01,1999,0.280443,0.030124,0.003097,-0.073002
1,1,000002,1999-02,7.68,2187266.33,2958771.28,-0.021656,0.250319,1999-02,1999,0.280443,0.030124,0.003097,-0.024753
2,2,000002,1999-03,7.84,2232834.38,3020412.35,0.020833,0.250319,1999-03,1999,0.280443,0.030124,0.003097,0.017736
3,3,000002,1999-04,9.09,2588834.75,3501983.19,0.159439,0.250319,1999-04,1999,0.280443,0.030124,0.003097,0.156342
4,4,000002,1999-05,11.25,3204003.41,4334137.62,0.237624,0.250319,1999-05,1999,0.280443,0.030124,0.003097,0.234527
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
637879,637879,920992,2023-08,7.44,207562.44,719678.15,-0.174251,0.101222,2023-08,2023,0.117393,0.016171,0.001365,-0.175616
637880,637880,920992,2023-09,7.28,203098.73,704201.20,-0.021505,0.101222,2023-09,2023,0.117393,0.016171,0.001283,-0.022788
637881,637881,920992,2023-10,6.61,184406.95,639391.47,-0.092033,0.101222,2023-10,2023,0.117393,0.016171,0.001283,-0.093316
637882,637882,920992,2023-11,9.12,254431.37,882186.12,0.379727,0.101222,2023-11,2023,0.117393,0.016171,0.001283,0.378444


In [5]:
fama_factors = pd.read_csv('C:/Users/刘子暄/Desktop/五因子模型指标(月)223836302(仅供vip使用)/STK_MKT_FIVEFACMONTH.csv')

In [6]:
# 筛选匹配条件（沪深A股+创业板、2*3组合、流通市值加权，与Li2025对齐）
fama_clean = fama_factors[
    (fama_factors["MarkettypeID"] == "P9709") &  # 沪深A股+创业板
    (fama_factors["Portfolios"] == 1)             # 2*3投资组合划分
][[
    "TradingMonth", "RiskPremium1", "SMB1", "HML1", "RMW1", "CMA1"
]].rename(columns={
    "TradingMonth": "year_month",       # 统一时间列名
    "RiskPremium1": "mkt_rf",           # 市场风险溢价（流通加权）
    "SMB1": "smb",                      # 市值因子（流通加权）
    "HML1": "hml",                      # 价值因子（流通加权）
    "RMW1": "rmw",                      # 盈利因子（流通加权）
    "CMA1": "cma"                       # 投资因子（流通加权）
})
# 时间格式统一（转为YYYY-MM字符串，与mnth_data对齐）
fama_clean["year_month"] = fama_clean["year_month"].astype(str)


# ========== 2. 处理mnth_data：提取月度超额收益+时间对齐 ==========
# 生成year_month列（与Fama因子匹配）
#mnth_data["year_month"] = mnth_data["trade_month"].dt.strftime("%Y-%m")
# 去重：确保每个(ts_code, year_month)唯一（避免同一股票同月多条数据）
mnth_data = mnth_data.drop_duplicates(subset=["ts_code", "year_month"]).reset_index(drop=True)
# 提取核心列：股票代码、时间、月度超额收益、年份
mnth_core = mnth_data[["ts_code", "year_month", "trade_month", "year", "monthly_excess_return"]]


# ========== 3. 匹配Fama因子与月度超额收益 ==========
# 按year_month合并，确保每一行有“股票月度收益+当月Fama因子”
data_merged = pd.merge(
    mnth_core,
    fama_clean,
    on="year_month",
    how="inner"  # 只保留有因子数据的月份
).dropna(subset=["monthly_excess_return"])  # 剔除收益缺失值
print(f"因子与收益匹配完成，有效样本数：{len(data_merged)}条")
print(f"时间范围：{data_merged['year_month'].min()} ~ {data_merged['year_month'].max()}")

因子与收益匹配完成，有效样本数：633429条
时间范围：1999-01 ~ 2023-12


In [7]:
data_merged

,ts_code,year_month,trade_month,year,monthly_excess_return,mkt_rf,smb,hml,rmw,cma
0,000002,1999-01,1999-01,1999,-0.073002,-0.011274,0.016015,-0.004331,-0.005472,-0.011130
1,000006,1999-01,1999-01,1999,-0.076711,-0.011274,0.016015,-0.004331,-0.005472,-0.011130
2,000007,1999-01,1999-01,1999,-0.033764,-0.011274,0.016015,-0.004331,-0.005472,-0.011130
3,000008,1999-01,1999-01,1999,0.447542,-0.011274,0.016015,-0.004331,-0.005472,-0.011130
4,000009,1999-01,1999-01,1999,-0.101948,-0.011274,0.016015,-0.004331,-0.005472,-0.011130
...,...,...,...,...,...,...,...,...,...,...
637879,603993,2016-06,2016-06,2016,0.073888,0.021352,0.051807,-0.011581,-0.014007,-0.016938
637880,603996,2016-06,2016-06,2016,0.019502,0.021352,0.051807,-0.011581,-0.014007,-0.016938
637881,603997,2016-06,2016-06,2016,0.069258,0.021352,0.051807,-0.011581,-0.014007,-0.016938
637882,603998,2016-06,2016-06,2016,0.105848,0.021352,0.051807,-0.011581,-0.014007,-0.016938


In [8]:
fama_clean

,year_month,mkt_rf,smb,hml,rmw,cma
2,1999-01,-0.011274,0.016015,-0.004331,-0.005472,-0.011130
26,1999-02,-0.045161,0.006741,0.002148,0.009381,0.002877
50,1999-03,0.065888,0.049416,0.044272,-0.083860,0.063389
74,1999-04,-0.035693,-0.016635,0.032332,-0.016524,0.013604
98,1999-05,0.109583,-0.037394,-0.017918,0.012674,-0.053147
...,...,...,...,...,...,...
7921,2023-08,-0.052180,0.014287,0.002432,-0.001924,0.003302
7965,2023-09,-0.003699,0.011300,0.018374,-0.014387,-0.001379
8015,2023-10,-0.027992,0.030112,-0.009732,-0.015968,0.001494
8058,2023-11,-0.000803,0.053392,-0.004801,-0.004537,0.019241


构建动量因子

In [9]:
co_data = pd.read_csv("C:/Users/刘子暄/Desktop/金融计量学pre2/TRD_Co.csv", encoding='utf-8')
co_data.rename(columns={
    'Stkcd': 'ts_code',
    'Stknme': 'name',
    'Listdt': 'list_date',
    'Indcd': 'industry_code'  #行业代码是0001-0006，这个字段很重要
}, inplace=True)

# 再确保股票代码是6位数字
co_data['ts_code'] = co_data['ts_code'].astype(str).str.zfill(6)

print("公司数据共有", len(co_data), "只股票")
co_data

公司数据共有 5352 只股票


,ts_code,name,list_date,industry_code
0,000002,万科A,1991-01-29,3
1,000006,深振业A,1992-04-27,3
2,000007,全新好,1992-04-13,3
3,000008,神州高铁,1992-05-07,5
4,000009,中国宝安,1991-06-25,4
...,...,...,...,...
5347,920978,开特股份,2023-09-28,5
5348,920981,晶赛科技,2021-11-15,5
5349,920982,锦波生物,2023-07-20,5
5350,920985,海泰新能,2022-08-08,5


In [10]:
#动量因子
mnth_sorted = mnth_data.sort_values(['ts_code', 'trade_month']).reset_index(drop=True)
# 步骤2：滚动计算过去11个月收益率均值（shift(1)排除当月，避免短期反转）
# 窗口：11个月，至少6个月数据才计算（月度频率下窗口为11个月度观测值）
mnth_sorted['momentum'] = mnth_sorted.groupby('ts_code')['monthly_excess_return'].rolling(
    window=11, min_periods=6
).mean().shift(1).reset_index(level=0, drop=True)  # 重置索引对齐
# 步骤3：提取月度动量值（每个股票每月1个值）
momentum_data = mnth_sorted[['ts_code', 'year_month', 'momentum']].drop_duplicates()
# 合并行业数据用于填充缺失值
momentum_data = pd.merge(
    momentum_data,
    co_data[['ts_code', 'industry_code']].drop_duplicates(),
    on='ts_code',
    how='left'
)
# 填充缺失值（行业-年月中位数）
momentum_data['momentum'] = momentum_data.groupby(['year_month', 'industry_code'])['momentum'].transform(
    lambda x: x.fillna(x.median())
)

D:\jus\envs\pytorch\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
D:\jus\envs\pytorch\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
D:\jus\envs\pytorch\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
D:\jus\envs\pytorch\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
D:\jus\envs\pytorch\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
D:\jus\envs\pytorch\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
D:\jus\envs\pytorch\li

In [11]:
momentum_data

,ts_code,year_month,momentum,industry_code
0,000002,1999-01,-0.004255,3
1,000002,1999-02,0.007928,3
2,000002,1999-03,-0.005390,3
3,000002,1999-04,-0.014522,3
4,000002,1999-05,NaN,3
...,...,...,...,...
637879,920992,2023-08,-0.048338,5
637880,920992,2023-09,-0.061065,5
637881,920992,2023-10,-0.057586,5
637882,920992,2023-11,-0.067800,5


In [12]:
data_merged=pd.merge(data_merged,momentum_data,on=['ts_code','year_month'],how='left')
#data_merged['momentum'].filna(method)


In [13]:
data_merged

,ts_code,year_month,trade_month,year,monthly_excess_return,mkt_rf,smb,hml,rmw,cma,momentum,industry_code
0,000002,1999-01,1999-01,1999,-0.073002,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.004255,3
1,000006,1999-01,1999-01,1999,-0.076711,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.044842,3
2,000007,1999-01,1999-01,1999,-0.033764,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.014817,3
3,000008,1999-01,1999-01,1999,0.447542,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.030936,5
4,000009,1999-01,1999-01,1999,-0.101948,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.003617,4
...,...,...,...,...,...,...,...,...,...,...,...,...
633424,603993,2016-06,2016-06,2016,0.073888,0.021352,0.051807,-0.011581,-0.014007,-0.016938,0.010494,5
633425,603996,2016-06,2016-06,2016,0.019502,0.021352,0.051807,-0.011581,-0.014007,-0.016938,-0.005359,5
633426,603997,2016-06,2016-06,2016,0.069258,0.021352,0.051807,-0.011581,-0.014007,-0.016938,-0.001808,5
633427,603998,2016-06,2016-06,2016,0.105848,0.021352,0.051807,-0.011581,-0.014007,-0.016938,-0.010574,5


table6 递归策略

In [14]:
full_features=pd.read_csv('features.csv')

In [15]:
full_features['ts_code'] = full_features['ts_code'].astype(str).str.zfill(6)

In [16]:
full_features

,Unnamed: 0,ts_code,year,industry_code,year_month,cash_over_total_assets,inventory_over_total_assets,fixed_assets_over_total_assets,total_assets_over_total_assets,total_liab_over_total_assets,...,cost_roll3_avg,gross_profit_roll3_avg,operating_profit_roll3_avg,rd_expense_roll3_avg,operate_cash_roll3_avg,invest_cash_roll3_avg,finance_cash_roll3_avg,profit_margin_roll3_avg,asset_turnover_roll3_avg,eq_ratio_roll3_avg
0,0,000002,1999,3.0,1999-01,0.107797,0.612378,0.048179,1.0,0.504005,...,8.999736e+08,9.336277e+07,1.086971e+08,0.0,-2.651082e+08,-67394259.34,2.238659e+08,0.087275,0.228796,0.495995
1,1,000002,1999,3.0,1999-02,0.107797,0.612378,0.048179,1.0,0.504005,...,8.999736e+08,9.336277e+07,1.086971e+08,0.0,-2.651082e+08,-67394259.34,2.238659e+08,0.087275,0.228796,0.495995
2,2,000002,1999,3.0,1999-03,0.107797,0.612378,0.048179,1.0,0.504005,...,8.999736e+08,9.336277e+07,1.086971e+08,0.0,-2.651082e+08,-67394259.34,2.238659e+08,0.087275,0.228796,0.495995
3,3,000002,1999,3.0,1999-04,0.107797,0.612378,0.048179,1.0,0.504005,...,8.999736e+08,9.336277e+07,1.086971e+08,0.0,-2.651082e+08,-67394259.34,2.238659e+08,0.087275,0.228796,0.495995
4,4,000002,1999,3.0,1999-05,0.107797,0.612378,0.048179,1.0,0.504005,...,8.999736e+08,9.336277e+07,1.086971e+08,0.0,-2.651082e+08,-67394259.34,2.238659e+08,0.087275,0.228796,0.495995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16982,16982,000151,2000,6.0,2000-03,0.676935,0.055269,0.037312,1.0,0.238420,...,2.986142e+08,6.899580e+07,7.169877e+07,0.0,6.190064e+07,-19676129.79,3.417208e+08,0.162903,0.531276,0.685542
16983,16983,000151,2000,6.0,2000-04,0.676935,0.055269,0.037312,1.0,0.238420,...,3.282039e+08,6.943889e+07,7.251740e+07,0.0,8.013330e+07,-16796125.40,5.057575e+08,0.158076,0.402027,0.761580
16984,16984,000151,2000,6.0,2000-05,0.676935,0.055269,0.037312,1.0,0.238420,...,3.282039e+08,6.943889e+07,7.251740e+07,0.0,8.013330e+07,-16796125.40,5.057575e+08,0.158076,0.402027,0.761580
16985,16985,000151,2000,6.0,2000-06,0.676935,0.055269,0.037312,1.0,0.238420,...,3.282039e+08,6.943889e+07,7.251740e+07,0.0,8.013330e+07,-16796125.40,5.057575e+08,0.158076,0.402027,0.761580


In [29]:
data_merged

,ts_code,year_month,trade_month,year,monthly_excess_return,mkt_rf,smb,hml,rmw,cma,momentum,industry_code,circ_mv_x,circ_mv_y
0,000002,1999-01,1999-01,1999,-0.073002,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.004255,3,2235682.38,2235682.38
1,000006,1999-01,1999-01,1999,-0.076711,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.044842,3,1529150.21,1529150.21
2,000007,1999-01,1999-01,1999,-0.033764,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.014817,3,578408.19,578408.19
3,000008,1999-01,1999-01,1999,0.447542,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.030936,5,440779.26,440779.26
4,000009,1999-01,1999-01,1999,-0.101948,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.003617,4,2270514.18,2270514.18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
633424,603993,2016-06,2016-06,2016,0.073888,0.021352,0.051807,-0.011581,-0.014007,-0.016938,0.010494,5,53757982.40,53757982.40
633425,603996,2016-06,2016-06,2016,0.019502,0.021352,0.051807,-0.011581,-0.014007,-0.016938,-0.005359,5,2292826.50,2292826.50
633426,603997,2016-06,2016-06,2016,0.069258,0.021352,0.051807,-0.011581,-0.014007,-0.016938,-0.001808,5,2250360.00,2250360.00
633427,603998,2016-06,2016-06,2016,0.105848,0.021352,0.051807,-0.011581,-0.014007,-0.016938,-0.010574,5,3951521.59,3951521.59


In [30]:
import pandas as pd

merged_data = pd.merge(full_features, data_merged, on=['ts_code', 'year_month', 'industry_code','year'], how='inner')

merged_data

,Unnamed: 0,ts_code,year,industry_code,year_month,cash_over_total_assets,inventory_over_total_assets,fixed_assets_over_total_assets,total_assets_over_total_assets,total_liab_over_total_assets,...,trade_month,monthly_excess_return,mkt_rf,smb,hml,rmw,cma,momentum,circ_mv_x,circ_mv_y
0,0,000002,1999,3.0,1999-01,0.107797,0.612378,0.048179,1.0,0.504005,...,1999-01,-0.073002,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.004255,2235682.38,2235682.38
1,1,000002,1999,3.0,1999-02,0.107797,0.612378,0.048179,1.0,0.504005,...,1999-02,-0.024753,-0.045161,0.006741,0.002148,0.009381,0.002877,0.007928,2187266.33,2187266.33
2,2,000002,1999,3.0,1999-03,0.107797,0.612378,0.048179,1.0,0.504005,...,1999-03,0.017736,0.065888,0.049416,0.044272,-0.083860,0.063389,-0.005390,2232834.38,2232834.38
3,3,000002,1999,3.0,1999-04,0.107797,0.612378,0.048179,1.0,0.504005,...,1999-04,0.156342,-0.035693,-0.016635,0.032332,-0.016524,0.013604,-0.014522,2588834.75,2588834.75
4,4,000002,1999,3.0,1999-05,0.107797,0.612378,0.048179,1.0,0.504005,...,1999-05,0.234527,0.109583,-0.037394,-0.017918,0.012674,-0.053147,NaN,3204003.41,3204003.41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16290,16975,000100,2023,5.0,2023-08,0.075936,0.050780,0.395407,1.0,0.627014,...,2023-08,-0.045654,-0.052180,0.014287,0.002432,-0.001924,0.003302,0.013488,74204020.37,74204020.37
16291,16976,000100,2023,5.0,2023-09,0.075936,0.050780,0.395407,1.0,0.627014,...,2023-09,-0.006161,-0.003699,0.011300,0.018374,-0.014387,-0.001379,0.020841,73842049.54,73842049.54
16292,16977,000100,2023,5.0,2023-10,0.057265,0.048273,0.460803,1.0,0.620576,...,2023-10,-0.038048,-0.027992,0.030112,-0.009732,-0.015968,0.001494,0.013393,71127268.31,71127268.31
16293,16978,000100,2023,5.0,2023-11,0.057265,0.048273,0.460803,1.0,0.620576,...,2023-11,0.062330,-0.000803,0.053392,-0.004801,-0.004537,0.019241,0.007500,75651903.70,75651903.70


In [39]:
#merged_data.drop(columns='circ_mv_y',inplace=True)
merged_data.rename(columns={'circ_mv_x': 'circ_mv'}, inplace=True)

In [31]:
merged_data.dropna(inplace=True)
merged_data.drop(columns='Unnamed: 0',inplace=True)

In [32]:
merged_data['year'] = merged_data['year_month'].astype(str).str[:4].astype(int)
merged_data['month'] = merged_data['year_month'].astype(str).str[5:].astype(int)
merged_data

,ts_code,year,industry_code,year_month,cash_over_total_assets,inventory_over_total_assets,fixed_assets_over_total_assets,total_assets_over_total_assets,total_liab_over_total_assets,total_eq_over_total_assets,...,monthly_excess_return,mkt_rf,smb,hml,rmw,cma,momentum,circ_mv_x,circ_mv_y,month
0,000002,1999,3.0,1999-01,0.107797,0.612378,0.048179,1.0,0.504005,0.495995,...,-0.073002,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.004255,2235682.38,2235682.38,1
1,000002,1999,3.0,1999-02,0.107797,0.612378,0.048179,1.0,0.504005,0.495995,...,-0.024753,-0.045161,0.006741,0.002148,0.009381,0.002877,0.007928,2187266.33,2187266.33,2
2,000002,1999,3.0,1999-03,0.107797,0.612378,0.048179,1.0,0.504005,0.495995,...,0.017736,0.065888,0.049416,0.044272,-0.083860,0.063389,-0.005390,2232834.38,2232834.38,3
3,000002,1999,3.0,1999-04,0.107797,0.612378,0.048179,1.0,0.504005,0.495995,...,0.156342,-0.035693,-0.016635,0.032332,-0.016524,0.013604,-0.014522,2588834.75,2588834.75,4
5,000002,1999,3.0,1999-06,0.107797,0.612378,0.048179,1.0,0.504005,0.495995,...,0.264459,0.355659,-0.078981,-0.089648,0.002915,0.032148,-0.011450,4061252.32,4061252.32,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16290,000100,2023,5.0,2023-08,0.075936,0.050780,0.395407,1.0,0.627014,0.372986,...,-0.045654,-0.052180,0.014287,0.002432,-0.001924,0.003302,0.013488,74204020.37,74204020.37,8
16291,000100,2023,5.0,2023-09,0.075936,0.050780,0.395407,1.0,0.627014,0.372986,...,-0.006161,-0.003699,0.011300,0.018374,-0.014387,-0.001379,0.020841,73842049.54,73842049.54,9
16292,000100,2023,5.0,2023-10,0.057265,0.048273,0.460803,1.0,0.620576,0.379424,...,-0.038048,-0.027992,0.030112,-0.009732,-0.015968,0.001494,0.013393,71127268.31,71127268.31,10
16293,000100,2023,5.0,2023-11,0.057265,0.048273,0.460803,1.0,0.620576,0.379424,...,0.062330,-0.000803,0.053392,-0.004801,-0.004537,0.019241,0.007500,75651903.70,75651903.70,11


In [33]:
features_c=full_features.columns.tolist()
del features_c[:5]
features_c

['cash_over_total_assets',
 'inventory_over_total_assets',
 'fixed_assets_over_total_assets',
 'total_assets_over_total_assets',
 'total_liab_over_total_assets',
 'total_eq_over_total_assets',
 'current_assets_over_total_assets',
 'current_liab_over_total_assets',
 'net_profit_over_total_assets',
 'revenue_over_total_assets',
 'cost_over_total_assets',
 'gross_profit_over_total_assets',
 'operating_profit_over_total_assets',
 'rd_expense_over_total_assets',
 'operate_cash_over_total_assets',
 'invest_cash_over_total_assets',
 'finance_cash_over_total_assets',
 'profit_margin_over_total_assets',
 'asset_turnover_over_total_assets',
 'eq_ratio_over_total_assets',
 'cash_yoy_growth',
 'inventory_yoy_growth',
 'fixed_assets_yoy_growth',
 'total_assets_yoy_growth',
 'total_liab_yoy_growth',
 'total_eq_yoy_growth',
 'current_assets_yoy_growth',
 'current_liab_yoy_growth',
 'net_profit_yoy_growth',
 'revenue_yoy_growth',
 'cost_yoy_growth',
 'gross_profit_yoy_growth',
 'operating_profit_yoy_g

In [40]:
merged_data

,ts_code,year,industry_code,year_month,cash_over_total_assets,inventory_over_total_assets,fixed_assets_over_total_assets,total_assets_over_total_assets,total_liab_over_total_assets,total_eq_over_total_assets,...,trade_month,monthly_excess_return,mkt_rf,smb,hml,rmw,cma,momentum,circ_mv,month
0,000002,1999,3.0,1999-01,0.107797,0.612378,0.048179,1.0,0.504005,0.495995,...,1999-01,-0.073002,-0.011274,0.016015,-0.004331,-0.005472,-0.011130,-0.004255,2235682.38,1
1,000002,1999,3.0,1999-02,0.107797,0.612378,0.048179,1.0,0.504005,0.495995,...,1999-02,-0.024753,-0.045161,0.006741,0.002148,0.009381,0.002877,0.007928,2187266.33,2
2,000002,1999,3.0,1999-03,0.107797,0.612378,0.048179,1.0,0.504005,0.495995,...,1999-03,0.017736,0.065888,0.049416,0.044272,-0.083860,0.063389,-0.005390,2232834.38,3
3,000002,1999,3.0,1999-04,0.107797,0.612378,0.048179,1.0,0.504005,0.495995,...,1999-04,0.156342,-0.035693,-0.016635,0.032332,-0.016524,0.013604,-0.014522,2588834.75,4
5,000002,1999,3.0,1999-06,0.107797,0.612378,0.048179,1.0,0.504005,0.495995,...,1999-06,0.264459,0.355659,-0.078981,-0.089648,0.002915,0.032148,-0.011450,4061252.32,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16290,000100,2023,5.0,2023-08,0.075936,0.050780,0.395407,1.0,0.627014,0.372986,...,2023-08,-0.045654,-0.052180,0.014287,0.002432,-0.001924,0.003302,0.013488,74204020.37,8
16291,000100,2023,5.0,2023-09,0.075936,0.050780,0.395407,1.0,0.627014,0.372986,...,2023-09,-0.006161,-0.003699,0.011300,0.018374,-0.014387,-0.001379,0.020841,73842049.54,9
16292,000100,2023,5.0,2023-10,0.057265,0.048273,0.460803,1.0,0.620576,0.379424,...,2023-10,-0.038048,-0.027992,0.030112,-0.009732,-0.015968,0.001494,0.013393,71127268.31,10
16293,000100,2023,5.0,2023-11,0.057265,0.048273,0.460803,1.0,0.620576,0.379424,...,2023-11,0.062330,-0.000803,0.053392,-0.004801,-0.004537,0.019241,0.007500,75651903.70,11


In [24]:
#merged_data['year_month'] = merged_data['year_month'].astype(int)
merged_data['year_month'] = pd.to_datetime(merged_data['year_month'], format='%Y-%m').dt.strftime('%Y%m').astype(int)

In [69]:
data

,pred_return,ts_code,year_month,monthly_excess_return,circ_mv,industry_code,year
0,0.007185,000002,199901,-0.073002,2235682.38,3.0,1999
1,0.014298,000006,199901,-0.076711,1529150.21,3.0,1999
2,0.005922,000007,199901,-0.033764,578408.19,3.0,1999
3,0.035980,000008,199901,0.447542,440779.26,5.0,1999
4,0.004567,000009,199901,-0.101948,2270514.18,4.0,1999
...,...,...,...,...,...,...,...
16192,0.000565,000089,202312,-0.057083,13186273.65,2.0,2023
16193,-0.003512,000090,202312,-0.093985,8594915.70,3.0,2023
16194,0.005240,000096,202312,0.056199,5920728.32,6.0,2023
16195,0.008191,000099,202312,0.104245,6249893.20,2.0,2023


In [47]:
pred_df

,Unnamed: 0,ts_code,year_month,pred_return,monthly_excess_return,circ_mv,finance_cash_lag1,total_eq_relative_industry,asset_turnover_relative_industry,cash_yoy_growth,...,cost_yoy_growth,revenue_yoy_growth,total_eq_yoy_growth,net_profit_over_total_assets,eq_ratio_relative_industry,rd_expense_roll3_avg,rd_expense_lag1,rd_expense_yoy_growth,profit_margin_relative_industry,asset_turnover_yoy_growth
0,0,000002,199901,0.007185,-0.073002,2235682.38,0.000000e+00,2.499470,0.754238,0.000000,...,0.00000,0.000000,0.000000,0.019968,0.917958,0.000000e+00,0.00,0.000000,21.876471,0.000000
1,1,000002,199902,0.007554,-0.024753,2187266.33,2.238659e+08,2.499470,0.754238,0.000000,...,0.00000,0.000000,0.000000,0.019968,0.917958,0.000000e+00,0.00,0.000000,21.876471,0.000000
2,2,000002,199903,0.007554,0.017736,2232834.38,2.238659e+08,2.499470,0.754238,0.000000,...,0.00000,0.000000,0.000000,0.019968,0.917958,0.000000e+00,0.00,0.000000,21.876471,0.000000
3,3,000002,199904,0.007554,0.156342,2588834.75,2.238659e+08,2.499470,0.754238,0.000000,...,0.00000,0.000000,0.000000,0.019968,0.917958,0.000000e+00,0.00,0.000000,21.876471,0.000000
4,4,000002,199905,0.007554,0.234527,3204003.41,2.238659e+08,2.499470,0.754238,0.000000,...,0.00000,0.000000,0.000000,0.019968,0.917958,0.000000e+00,0.00,0.000000,21.876471,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
633313,637768,920992,202308,0.021729,-0.175616,207562.44,-2.308448e+07,0.081813,0.775864,0.000000,...,0.00000,0.000000,0.000000,0.016683,1.305290,2.251858e+07,25556137.44,0.000000,-0.045986,0.000000
633314,637769,920992,202309,0.020915,-0.022788,203098.73,-2.308448e+07,0.081813,0.775864,0.000000,...,0.00000,0.000000,0.000000,0.016683,1.305290,2.555614e+07,25556137.44,0.000000,-0.045986,0.000000
633315,637770,920992,202310,-0.009616,-0.093316,184406.95,-2.308448e+07,0.082361,1.048111,1.161852,...,0.33801,0.341625,0.006699,0.022006,1.323114,2.814918e+07,25556137.44,0.304394,-0.044902,0.350895
633316,637771,920992,202311,0.011922,0.378444,254431.37,-2.474023e+07,0.082361,1.048111,0.000000,...,0.00000,0.000000,0.000000,0.022006,1.323114,3.074223e+07,33335275.72,0.000000,-0.044902,0.000000


30个信号的递归策略

In [65]:
# ========== Newey-West标准误计算 ========== 
def newey_west_t_stat(series, lags=12):
    """计算Newey-West调整的t统计量（滞后12期）"""
    series = pd.Series(series).dropna()
    if len(series) < lags + 1:
        return np.nan
    
    X = np.ones(len(series))
    try:
        model = sm.OLS(series.values, X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
        return model.tvalues[0]
    except:
        return np.nan

def newey_west_regression(y, X, lags=12):
    """进行Newey-West调整的回归"""
    try:
        if len(y) < lags + 1:
            return None, None
        
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        
        X_const = sm.add_constant(X)
        model = sm.OLS(y, X_const).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
        return model.params, model.tvalues
    except Exception as e:
        print(f"回归错误: {e}")
        return None, None

# ========== 计算每个信号的月度多空组合收益 ==========
def calculate_signal_long_short_returns(merged_data, signal_list, weighting='equal'):
    """
    计算每个信号的月度多空组合收益
    weighting: 'equal' 等权, 'value' 市值加权
    """
    print(f"计算每个信号的月度多空组合收益 ({weighting}加权)...")
    
    results = {}
    
    for i, signal in enumerate(signal_list):
        if i % 10 == 0:
            print(f"  正在处理信号 {i+1}/{len(signal_list)}: {signal}")
        
        try:
            data = merged_data.copy()
            data = data.dropna(subset=[signal, 'monthly_excess_return', 'circ_mv'])
            
            if len(data) < 100:
                continue
            
            monthly_ls_returns = []
            
            for month in sorted(data['year_month'].unique()):
                month_data = data[data['year_month'] == month].copy()
                
                if len(month_data) < 50:
                    continue
                
                # 按信号排序（升序，确保后续分组正确）
                month_data['signal_rank'] = month_data[signal].rank(pct=True)
                
                # 前10%（信号最大，做多）和后10%（信号最小，做空）
                long_portfolio = month_data[month_data['signal_rank'] >= 0.9]
                short_portfolio = month_data[month_data['signal_rank'] <= 0.1]
                
                if len(long_portfolio) > 0 and len(short_portfolio) > 0:
                    if weighting == 'equal':
                        # 等权组合
                        long_return = long_portfolio['monthly_excess_return'].mean()
                        short_return = short_portfolio['monthly_excess_return'].mean()
                    else:
                        # 市值加权组合
                        long_total_mv = long_portfolio['circ_mv'].sum()
                        short_total_mv = short_portfolio['circ_mv'].sum()
                        
                        if long_total_mv > 0 and short_total_mv > 0:
                            long_return = (long_portfolio['monthly_excess_return'] * 
                                         long_portfolio['circ_mv']).sum() / long_total_mv
                            short_return = (short_portfolio['monthly_excess_return'] * 
                                          short_portfolio['circ_mv']).sum() / short_total_mv
                        else:
                            continue
                    
                    ls_return = long_return - short_return
                    
                    monthly_ls_returns.append({
                        'year_month': month,
                        'signal': signal,
                        'long_return': long_return,
                        'short_return': short_return,
                        'ls_return': ls_return,
                        'n_long': len(long_portfolio),
                        'n_short': len(short_portfolio)
                    })
            
            if monthly_ls_returns:
                results[signal] = pd.DataFrame(monthly_ls_returns)
                
        except Exception as e:
            print(f"  信号 {signal} 计算失败: {e}")
            continue
    
    print(f"  完成！共计算了{len(results)}个信号的月度多空组合收益")
    return results

# ========== 计算滚动窗口的t统计量 ==========
def calculate_rolling_t_statistics(signal_returns_dict, window=36):
    """计算每个信号滚动窗口的t统计量"""
    print("计算滚动窗口的t统计量...")
    
    t_stats_dict = {}
    
    for i, (signal, returns_df) in enumerate(signal_returns_dict.items()):
        if i % 10 == 0:
            print(f"  正在处理信号 {i+1}/{len(signal_returns_dict)}")
        
        returns_df = returns_df.sort_values('year_month')
        
        t_stats = []
        dates = []
        
        # 滚动计算t统计量
        for j in range(window, len(returns_df)):
            window_data = returns_df.iloc[j-window:j]
            
            if len(window_data) >= 24:
                ls_returns = window_data['ls_return'].values
                t_stat = newey_west_t_stat(ls_returns, lags=12)
                
                if not np.isnan(t_stat):
                    t_stats.append(t_stat)
                    dates.append(returns_df.iloc[j]['year_month'])
        
        if t_stats:
            t_stats_dict[signal] = pd.DataFrame({
                'year_month': dates,
                'signal': signal,
                't_stat': t_stats
            })
    
    print(f"  完成！共计算了{len(t_stats_dict)}个信号的滚动t统计量")
    return t_stats_dict

# ========== 构建递归排序投资组合==========
def form_recursive_ranking_portfolio(signal_returns_dict, t_stats_dict, weighting='equal',
                                    rebalancing_freq='annual', n_groups=10):
    """
    weighting: 'equal' 等权, 'value' 市值加权
    n_groups: 分组数量，默认为10组
    """
    print(f"构建递归排序投资组合 ({weighting}加权, {n_groups}组，按历史平均收益排序)...")

    # 准备数据
    all_signal_returns = []
    for signal, returns_df in signal_returns_dict.items():
        if signal in t_stats_dict:  # 仍要求该信号有t统计量（保证信号质量）
            all_signal_returns.append(returns_df[['year_month', 'signal', 'ls_return']])
    
    if not all_signal_returns:
        print("  错误：没有可用的信号收益数据")
        return pd.DataFrame(), pd.DataFrame()
    
    signal_returns_all = pd.concat(all_signal_returns, ignore_index=True)
    
    # 获取所有再平衡月份（每年6月）
    all_months = sorted(signal_returns_all['year_month'].unique())
    rebalance_months = [m for m in all_months if str(m).endswith('06')]
    print(f"  找到{len(rebalance_months)}个再平衡月份")

    # 存储分组收益
    group_returns_dict = {i: [] for i in range(1, n_groups + 1)}
    portfolio_returns = []

    for i, rebalance_month in enumerate(rebalance_months):
        if i == 0:
            continue  # 第一个再平衡月无足够历史数据

        prev_rebalance = rebalance_months[i - 1]

        # === 关键修改：按历史平均 ls_return 排序 ===
        signal_avg_returns = []
        for signal in signal_returns_all['signal'].unique():
            hist = signal_returns_all[
                (signal_returns_all['signal'] == signal) &
                (signal_returns_all['year_month'] <= prev_rebalance)
            ]['ls_return']
            if len(hist) >= 12:  # 至少12个月历史
                signal_avg_returns.append({'signal': signal, 'avg_ls_return': hist.mean()})
        
        if len(signal_avg_returns) < n_groups:
            continue  # 信号太少，跳过

        # 按平均收益升序排序：第1组 = 最低收益，第10组 = 最高收益
        signal_ranking = pd.DataFrame(signal_avg_returns).sort_values('avg_ls_return', ascending=True)
        n_signals = len(signal_ranking)
        signals_per_group = max(1, n_signals // n_groups)

        # 分组
        grouped_signals = {}
        for group in range(1, n_groups + 1):
            start = (group - 1) * signals_per_group
            end = group * signals_per_group if group < n_groups else n_signals
            grouped_signals[group] = signal_ranking.iloc[start:end]['signal'].tolist()

        # 获取接下来12个月的收益
        months_after = sorted([m for m in all_months if m > rebalance_month])[:12]

        for month in months_after:
            month_data = signal_returns_all[signal_returns_all['year_month'] == month]
            if month_data.empty:
                continue

            group_month_returns = {}
            for group, signals in grouped_signals.items():
                returns = month_data[month_data['signal'].isin(signals)]['ls_return']
                if len(returns) > 0:
                    group_month_returns[group] = returns.mean()

            # 记录分组收益
            for group, ret in group_month_returns.items():
                group_returns_dict[group].append({
                    'year_month': month,
                    'rebalance_month': rebalance_month,
                    'group': group,
                    'return': ret
                })

            # 多空组合：10 - 1
            if 1 in group_month_returns and 10 in group_month_returns:
                ls_ret = group_month_returns[10] - group_month_returns[1]
                portfolio_returns.append({
                    'year_month': month,
                    'rebalance_month': rebalance_month,
                    'portfolio_return': ls_ret
                })

    # 转换为DataFrame
    all_group_returns = []
    for group, returns_list in group_returns_dict.items():
        if returns_list:
            all_group_returns.append(pd.DataFrame(returns_list))
    
    group_returns_df = pd.concat(all_group_returns, ignore_index=True) if all_group_returns else pd.DataFrame()
    portfolio_returns_df = pd.DataFrame(portfolio_returns)

    return group_returns_df, portfolio_returns_df

# ========== 计算分组完整统计量（含夏普比率） ==========
def calculate_group_full_statistics(group_returns_df, portfolio_returns_df):
    """计算分组完整统计量（含收益率Alpha、t-stat、夏普比率），并强制按 Avg_Return(%) 从小到大重排组号"""
    print("计算分组完整统计量（按 Avg 重排组号）...")

    results = []

    # 分组收益统计
    for group in sorted(group_returns_df['group'].unique()):
        group_data = group_returns_df[group_returns_df['group'] == group]
        returns = group_data['return'].dropna()

        if len(returns) > 0:
            mean_return = returns.mean() * 100  # 转换为百分比
            t_stat = newey_west_t_stat(returns, lags=12)
            std_dev = returns.std() * 100
            sharpe_ratio = (mean_return / std_dev) * np.sqrt(12) if std_dev != 0 else np.nan

            results.append({
                'Original_Group': int(group),  # 保留原始组号（可选）
                'Avg_Return(%)': mean_return,
                'Return_t-stat': t_stat,
                'Std_Dev(%)': std_dev,
                'SR': sharpe_ratio,
                'N': len(returns)
            })

    # 按 Avg_Return(%) 从小到大排序，并重新分配 Group 1~10
    if results:
        temp_df = pd.DataFrame(results).sort_values('Avg_Return(%)').reset_index(drop=True)
        temp_df['Group'] = range(1, len(temp_df) + 1)
    else:
        temp_df = pd.DataFrame()

    # 多空组合（10-1）单独处理：始终放在最后，不参与重排
    ls_row = None
    if not portfolio_returns_df.empty:
        ls_returns = portfolio_returns_df['portfolio_return'].dropna()
        if len(ls_returns) > 0:
            mean_return = ls_returns.mean() * 100
            t_stat = newey_west_t_stat(ls_returns, lags=12)
            std_dev = ls_returns.std() * 100
            sharpe_ratio = (mean_return / std_dev) * np.sqrt(12) if std_dev != 0 else np.nan

            ls_row = {
                'Group': '10-1',
                'Avg_Return(%)': mean_return,
                'Return_t-stat': t_stat,
                'Std_Dev(%)': std_dev,
                'SR': sharpe_ratio,
                'N': len(ls_returns)
            }

    # 合并分组与多空
    final_results = []
    if not temp_df.empty:
        for _, row in temp_df.iterrows():
            final_results.append({
                'Group': row['Group'],
                'Avg_Return(%)': row['Avg_Return(%)'],
                'Return_t-stat': row['Return_t-stat'],
                'Std_Dev(%)': row['Std_Dev(%)'],
                'SR': row['SR'],
                'N': row['N']
            })
    
    if ls_row is not None:
        final_results.append(ls_row)

    results_df = pd.DataFrame(final_results)
    return results_df

# ========== 计算因子调整Alpha ==========
def calculate_factor_alpha(returns_series, factor_data, model):
    """计算因子调整Alpha"""
    # 合并因子数据
    merged_data = pd.merge(
        returns_series,
        factor_data,
        on='year_month',
        how='inner'
    ).dropna()
    
    if len(merged_data) == 0:
        return np.nan, np.nan
    
    # 准备因子
    if model == 'CAPM':
        factors = ['mkt_rf']
    elif model == 'FF3':
        factors = ['mkt_rf', 'smb', 'hml']
    elif model == 'FF5':
        factors = ['mkt_rf', 'smb', 'hml', 'rmw', 'cma']
    elif model == 'FF5+MOM':
        factors = ['mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'momentum']
    else:
        return np.nan, np.nan
    
    # 检查因子是否存在
    missing_factors = [f for f in factors if f not in merged_data.columns]
    if missing_factors:
        print(f"  缺少因子: {missing_factors}")
        return np.nan, np.nan
    
    # 提取数据
    X = merged_data[factors].values
    y = merged_data['return'].values
    
    # Newey-West回归
    params, tvalues = newey_west_regression(y, X, lags=12)
    
    if params is not None and tvalues is not None:
        return params[0] * 100, tvalues[0]  # Alpha转换为百分数
    else:
        return np.nan, np.nan

def calculate_all_factor_alphas(group_returns_df, portfolio_returns_df, factor_data):
    """计算所有因子模型的Alpha（返回宽格式，便于合并）"""
    print("计算因子调整Alpha...")
    
    models = ['CAPM', 'FF3', 'FF5', 'FF5+MOM']
    results = []
    
    # 计算分组Alpha
    for group in sorted(group_returns_df['group'].unique()):
        group_data = group_returns_df[group_returns_df['group'] == group]
        
        if len(group_data) == 0:
            continue
        
        returns_series = group_data[['year_month', 'return']]
        row = {'Group': int(group)}
        
        for model in models:
            alpha, t_stat = calculate_factor_alpha(returns_series, factor_data, model)
            row[f'{model}_Alpha(%)'] = alpha
            row[f'{model}_t-stat'] = t_stat
        
        results.append(row)
    
    # 计算多空组合Alpha
    if not portfolio_returns_df.empty:
        ls_series = portfolio_returns_df[['year_month', 'portfolio_return']].rename(
            columns={'portfolio_return': 'return'}
        )
        row = {'Group': '10-1'}
        
        for model in models:
            alpha, t_stat = calculate_factor_alpha(ls_series, factor_data, model)
            row[f'{model}_Alpha(%)'] = alpha
            row[f'{model}_t-stat'] = t_stat
        
        results.append(row)
    
    # 转换为DataFrame并按Group排序
    results_df = pd.DataFrame(results)
    results_df['Group_sort'] = results_df['Group'].apply(lambda x: int(x) if str(x).isdigit() else 11)
    results_df = results_df.sort_values('Group_sort').drop('Group_sort', axis=1).reset_index(drop=True)
    
    return results_df

# ========== 合并统计量和Alpha表格（生成最终Table 6格式） ==========
def merge_statistics_and_alphas(statistics_df, alphas_df):
    """合并分组统计量和因子Alpha表格，生成原文格式"""
    # 合并两个DataFrame（按Group对齐）
    final_df = pd.merge(
        statistics_df[['Group', 'Avg_Return(%)', 'Return_t-stat', 'SR']],
        alphas_df,
        on='Group',
        how='inner'
    )
    
    # 调整列顺序为原文样式：Group → 收益率指标 → 4个因子模型的Alpha/t-stat
    columns_order = [
        'Group',
        'Avg_Return(%)', 'Return_t-stat', 'SR',
        'CAPM_Alpha(%)', 'CAPM_t-stat',
        'FF3_Alpha(%)', 'FF3_t-stat',
        'FF5_Alpha(%)', 'FF5_t-stat',
        'FF5+MOM_Alpha(%)', 'FF5+MOM_t-stat'
    ]
    
    # 确保所有列存在
    final_df = final_df.reindex(columns=columns_order, fill_value=np.nan)
    
    # 重命名列（匹配原文简洁格式）
    final_df.columns = [
        'Group',
        'Avg', 't-stat', 'SR',
        'CAPM_α', 'CAPM_t',
        'FF3_α', 'FF3_t',
        'FF5_α', 'FF5_t',
        'FF5+MOM_α', 'FF5+MOM_t'
    ]
    
    return final_df.round(4)

# ========== Table 6主函数 ==========
def reproduce_table6(merged_data, signal_list, factor_data):
    """复现Table 6：递归排序策略结果（等权+市值加权两张表格）"""
    print("="*80)
    print("开始复现Table 6：递归排序策略")
    print("="*80)
    
    # 确保有市值数据
    if 'circ_mv' not in merged_data.columns:
        print("错误：缺少流通市值数据(circ_mv)")
        return None, None
    
    # 1. 计算等权多空组合收益
    print("\n1. 计算等权多空组合收益")
    eq_signal_returns = calculate_signal_long_short_returns(merged_data, signal_list, weighting='equal')
    
    # 2. 计算市值加权多空组合收益
    print("\n2. 计算市值加权多空组合收益")
    vw_signal_returns = calculate_signal_long_short_returns(merged_data, signal_list, weighting='value')
    
    if not eq_signal_returns or not vw_signal_returns:
        print("错误：没有计算出任何信号的收益")
        return None, None
    
    # 3. 计算t统计量（使用等权收益计算，两种加权方式共用信号排序）
    print("\n3. 计算滚动窗口t统计量")
    t_stats_dict = calculate_rolling_t_statistics(eq_signal_returns, window=36)
    
    if not t_stats_dict:
        print("错误：没有计算出任何信号的t统计量")
        return None, None
    
    # 4. 构建原始分组组合（任意分组逻辑均可，例如按 t-stat 或历史收益）
    print("\n4. 构建原始等权投资组合")
    eq_group_raw, _ = form_recursive_ranking_portfolio(
        eq_signal_returns, t_stats_dict, weighting='equal', n_groups=10
    )

    print("\n5. 构建原始市值加权投资组合")
    vw_group_raw, _ = form_recursive_ranking_portfolio(
        vw_signal_returns, t_stats_dict, weighting='value', n_groups=10
    )

    # 6. 按实际 Avg 重排组号，并重新计算 10-1 多空收益
    print("\n6. 按实际平均收益重排组别并重建多空组合")
    eq_group_returns, eq_portfolio_returns = reorder_groups_by_avg_return(eq_group_raw)
    vw_group_returns, vw_portfolio_returns = reorder_groups_by_avg_return(vw_group_raw)

    if eq_group_returns.empty or vw_group_returns.empty:
        print("错误：重排后数据为空")
        return None, None
    
    # 6. 计算分组完整统计量（含夏普比率）
    print("\n6. 计算分组完整统计量")
    eq_statistics = calculate_group_full_statistics(eq_group_returns, eq_portfolio_returns)
    vw_statistics = calculate_group_full_statistics(vw_group_returns, vw_portfolio_returns)
    
    # 7. 计算因子调整Alpha（宽格式）
    print("\n7. 计算因子调整Alpha")
    eq_alphas = calculate_all_factor_alphas(eq_group_returns, eq_portfolio_returns, factor_data)
    vw_alphas = calculate_all_factor_alphas(vw_group_returns, vw_portfolio_returns, factor_data)
    
    # 8. 合并表格生成最终格式
    print("\n8. 合并表格生成最终结果")
    eq_final = merge_statistics_and_alphas(eq_statistics, eq_alphas)
    vw_final = merge_statistics_and_alphas(vw_statistics, vw_alphas)
    
    # 保存中间结果
    eq_group_returns.to_csv('table6_eq_group_returns.csv', index=False)
    vw_group_returns.to_csv('table6_vw_group_returns.csv', index=False)
    
    return eq_final, vw_final


def reorder_groups_by_avg_return(group_returns_df):
    """
    根据实际回测平均收益，将原始组别重映射为新组号（1=最小 Avg，10=最大 Avg），
    并基于新组别重新计算 10-1 多空组合的月度收益。
    
    输入：
        group_returns_df: 列包括 ['year_month', 'rebalance_month', 'group', 'return']
    输出：
        group_returns_df_new: 重映射后的长格式数据
        portfolio_returns_df_new: 新的 10-1 多空收益（仅含 ['year_month', 'rebalance_month', 'portfolio_return']）
    """
    if group_returns_df.empty:
        return group_returns_df, pd.DataFrame()

    # 1. 计算每组的平均收益，并按升序排序
    avg_by_group = group_returns_df.groupby('group')['return'].mean().sort_values()
    if len(avg_by_group) != 10:
        print(f"警告：组数不是10组（当前{len(avg_by_group)}组），跳过重排")
        return group_returns_df, pd.DataFrame()

    # 2. 构建旧组号 → 新组号（1~10）映射
    old_to_new = {old: new for new, old in enumerate(avg_by_group.index, start=1)}

    # 3. 重映射组别
    df_new = group_returns_df.copy()
    df_new['group'] = df_new['group'].map(old_to_new)

    # 4. 重构多空组合：10 - 1（基于新组号的月度收益）
    # 先转成宽表（month × group）
    pivot = df_new.pivot_table(
        index=['year_month', 'rebalance_month'],
        columns='group',
        values='return',
        aggfunc='mean'
    ).reset_index()

    # 确保 Group 1 和 10 都存在
    if 1 in pivot.columns and 10 in pivot.columns:
        pivot['portfolio_return'] = pivot[10] - pivot[1]
        portfolio_df = pivot[['year_month', 'rebalance_month', 'portfolio_return']].dropna()
    else:
        portfolio_df = pd.DataFrame()

    # 5. 返回长格式组收益 + 新多空收益
    group_long = df_new[['year_month', 'rebalance_month', 'group', 'return']].reset_index(drop=True)
    return group_long, portfolio_df



# ========== 主程序 ==========
if __name__ == "__main__":
    # 30个筛选出的信号
    selected_signals = [
        'finance_cash_lag1',
        'total_eq_relative_industry',
        'asset_turnover_relative_industry',
        'cash_yoy_growth',
        'inventory_roll3_avg',
        'operating_profit_roll3_avg',
        'gross_profit_roll3_avg',
        'inventory_over_total_assets',
        'cash_over_total_assets',
        'cash_relative_industry',
        'profit_margin_roll3_avg',
        'operate_cash_over_total_assets',
        'operating_profit_relative_industry',
        'finance_cash_over_total_assets',
        'profit_margin_over_total_assets',
        'current_assets_over_total_assets',
        'eq_ratio_over_total_assets',
        'cost_relative_industry',
        'total_assets_relative_industry',
        'rd_expense_relative_industry',
        'cost_yoy_growth',
        'revenue_yoy_growth',
        'total_eq_yoy_growth',
        'net_profit_over_total_assets',
        'eq_ratio_relative_industry',
        'rd_expense_roll3_avg',
        'rd_expense_lag1',
        'rd_expense_yoy_growth',
        'profit_margin_relative_industry',
        'asset_turnover_yoy_growth'
    ]
    
    print(f"使用{len(selected_signals)}个筛选出的信号")
    
    # 准备因子数据（自动适配常见列名）
    factor_columns = ['year_month', 'mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'momentum']
    merged_data = merged_data.copy()  # 确保merged_data已在外部定义
    
    # 因子列名映射（适配常见别名）
    factor_aliases = {
        'mkt_rf': ['mkt', 'market_return', 'excess_return'],
        'smb': ['SMB', 'size_factor'],
        'hml': ['HML', 'value_factor'],
        'rmw': ['RMW', 'profitability_factor'],
        'cma': ['CMA', 'investment_factor'],
        'momentum': ['MOM', 'umd', 'momentum_factor']
    }
    
    # 自动匹配因子列
    for target, aliases in factor_aliases.items():
        if target not in merged_data.columns:
            for alias in aliases:
                if alias in merged_data.columns:
                    merged_data[target] = merged_data[alias]
                    print(f"  自动匹配：{alias} → {target}")
                    break
    
    # 提取因子数据（去重去空）
    factor_data = merged_data[factor_columns].drop_duplicates().dropna()
    print(f"因子数据: {len(factor_data)}个月度，样本期间: {factor_data['year_month'].min()} - {factor_data['year_month'].max()}")
    
    # 运行Table 6复现
    eq_table, vw_table = reproduce_table6(
        merged_data, selected_signals, factor_data
    )
    
    # 输出最终结果（原文格式）
    if eq_table is not None and vw_table is not None:
        print("\n" + "="*100)
        print("Table 6 复现结果 - 等权组合（对应原文Panel A）")
        print("="*100)
        print(eq_table.to_string(index=False))
        
        print("\n" + "="*100)
        print("Table 6 复现结果 - 市值加权组合（对应原文Panel B）")
        print("="*100)
        print(vw_table.to_string(index=False))
        
        # 保存最终表格
        #eq_table.to_csv('table6_equal_weight_final.csv', index=False, encoding='utf-8-sig')
        #vw_table.to_csv('table6_value_weight_final.csv', index=False, encoding='utf-8-sig')
        print("\n最终表格已保存为CSV文件")
    else:
        print("\n复现失败，请检查数据或参数设置")

使用30个筛选出的信号
因子数据: 16043个月度，样本期间: 199901 - 202312
开始复现Table 6：递归排序策略

1. 计算等权多空组合收益
计算每个信号的月度多空组合收益 (equal加权)...
  正在处理信号 1/30: finance_cash_lag1
  正在处理信号 11/30: profit_margin_roll3_avg
  正在处理信号 21/30: cost_yoy_growth
  完成！共计算了30个信号的月度多空组合收益

2. 计算市值加权多空组合收益
计算每个信号的月度多空组合收益 (value加权)...
  正在处理信号 1/30: finance_cash_lag1
  正在处理信号 11/30: profit_margin_roll3_avg
  正在处理信号 21/30: cost_yoy_growth
  完成！共计算了30个信号的月度多空组合收益

3. 计算滚动窗口t统计量
计算滚动窗口的t统计量...
  正在处理信号 1/30
  正在处理信号 11/30
  正在处理信号 21/30
  完成！共计算了29个信号的滚动t统计量

4. 构建原始等权投资组合
构建递归排序投资组合 (equal加权, 10组，按历史平均收益排序)...
  找到23个再平衡月份

5. 构建原始市值加权投资组合
构建递归排序投资组合 (value加权, 10组，按历史平均收益排序)...
  找到23个再平衡月份

6. 按实际平均收益重排组别并重建多空组合

6. 计算分组完整统计量
计算分组完整统计量（按 Avg 重排组号）...
计算分组完整统计量（按 Avg 重排组号）...

7. 计算因子调整Alpha
计算因子调整Alpha...
计算因子调整Alpha...

8. 合并表格生成最终结果

Table 6 复现结果 - 等权组合（对应原文Panel A）
Group     Avg  t-stat      SR  CAPM_α  CAPM_t   FF3_α   FF3_t   FF5_α   FF5_t  FF5+MOM_α  FF5+MOM_t
  1.0 -0.6714 -1.4963 -0.2975 -0.6103 -2.8028 -0.2544 -1.2710 -0.8229 

100个信号的递归策略

In [41]:
# ========== 主程序 ==========
if __name__ == "__main__":
    # 100个信号
    selected_signals = features_c
    
    print(f"使用{len(features_c)}个筛选出的信号")
    
    # 准备因子数据（自动适配常见列名）
    factor_columns = ['year_month', 'mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'momentum']
    merged_data = merged_data.copy()  # 确保merged_data已在外部定义
    
    # 因子列名映射（适配常见别名）
    factor_aliases = {
        'mkt_rf': ['mkt', 'market_return', 'excess_return'],
        'smb': ['SMB', 'size_factor'],
        'hml': ['HML', 'value_factor'],
        'rmw': ['RMW', 'profitability_factor'],
        'cma': ['CMA', 'investment_factor'],
        'momentum': ['MOM', 'umd', 'momentum_factor']
    }
    
    # 自动匹配因子列
    for target, aliases in factor_aliases.items():
        if target not in merged_data.columns:
            for alias in aliases:
                if alias in merged_data.columns:
                    merged_data[target] = merged_data[alias]
                    print(f"  自动匹配：{alias} → {target}")
                    break
    
    # 提取因子数据（去重去空）
    factor_data = merged_data[factor_columns].drop_duplicates().dropna()
    print(f"因子数据: {len(factor_data)}个月度，样本期间: {factor_data['year_month'].min()} - {factor_data['year_month'].max()}")
    
    # 运行Table 6复现
    eq_table, vw_table = reproduce_table6(
        merged_data, selected_signals, factor_data
    )
    
    # 输出最终结果（原文格式）
    if eq_table is not None and vw_table is not None:
        print("\n" + "="*100)
        print("Table 6 复现结果 - 等权组合（对应原文Panel A）")
        print("="*100)
        print(eq_table.to_string(index=False))
        
        print("\n" + "="*100)
        print("Table 6 复现结果 - 市值加权组合（对应原文Panel B）")
        print("="*100)
        print(vw_table.to_string(index=False))
        
        # 保存最终表格
        #eq_table.to_csv('table6_equal_weight_final.csv', index=False, encoding='utf-8-sig')
        #vw_table.to_csv('table6_value_weight_final.csv', index=False, encoding='utf-8-sig')
        print("\n最终表格已保存为CSV文件")
    else:
        print("\n复现失败，请检查数据或参数设置")

使用100个筛选出的信号
因子数据: 16043个月度，样本期间: 1999-01 - 2023-12
开始复现Table 6：递归排序策略

1. 计算等权多空组合收益
计算每个信号的月度多空组合收益 (equal加权)...
  正在处理信号 1/100: cash_over_total_assets
  正在处理信号 11/100: cost_over_total_assets
  正在处理信号 21/100: cash_yoy_growth
  正在处理信号 31/100: cost_yoy_growth
  正在处理信号 41/100: cash_lag1
  正在处理信号 51/100: cost_lag1
  正在处理信号 61/100: cash_relative_industry
  正在处理信号 71/100: cost_relative_industry
  正在处理信号 81/100: cash_roll3_avg
  正在处理信号 91/100: cost_roll3_avg
  完成！共计算了99个信号的月度多空组合收益

2. 计算市值加权多空组合收益
计算每个信号的月度多空组合收益 (value加权)...
  正在处理信号 1/100: cash_over_total_assets
  正在处理信号 11/100: cost_over_total_assets
  正在处理信号 21/100: cash_yoy_growth
  正在处理信号 31/100: cost_yoy_growth
  正在处理信号 41/100: cash_lag1
  正在处理信号 51/100: cost_lag1
  正在处理信号 61/100: cash_relative_industry
  正在处理信号 71/100: cost_relative_industry
  正在处理信号 81/100: cash_roll3_avg
  正在处理信号 91/100: cost_roll3_avg
  完成！共计算了99个信号的月度多空组合收益

3. 计算滚动窗口t统计量
计算滚动窗口的t统计量...
  正在处理信号 1/99
  正在处理信号 11/99
  正在处理信号 21/99
  正在处理信号 31/99
  正在处理信号 41/99
 